In [8]:
import torch
import torch.nn as nn
import math

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model should be divisible by num_heads, which is currently not the case'

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(self.d_model, self.d_model)
        self.W_k = nn.Linear(self.d_model, self.d_model)
        self.W_v = nn.Linear(self.d_model, self.d_model)
        self.W_o = nn.Linear(self.d_model, self.d_model)

    def calculate_scaled_dot_product_attention(self, Q, K, V, mask=None):
        attention_scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_k)

        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e9)

        attention_probabilities = torch.softmax(attention_scores, dim=-1)
        contextual_embeddings = torch.matmul(attention_probabilities, V)

        return contextual_embeddings

    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, query_input, key_input, value_input, mask=None):
        Q = self.split_heads(self.W_q(query_input))
        K = self.split_heads(self.W_k(key_input))
        V = self.split_heads(self.W_v(value_input))

        multihead_contextual_embeddings = self.calculate_scaled_dot_product_attention(Q, K, V, mask)

        contextual_embeddings = self.W_o(self.combine_heads(multihead_contextual_embeddings))
        return contextual_embeddings

In [3]:
class PositionalFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super().__init__()

        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.pow(10000.0, torch.arange(0, d_model, 2, dtype=torch.float) / d_model)

        pe[:, 0::2] = torch.sin(position / div_term)
        pe[:, 1::2] = torch.cos(position / div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]



note = '''
Many resources use:
    div_term = torch.exp(torch.arange(0, d_model, 2, torch.float()) * -math.log(10000) / d_model)
Then:
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

This is the exact same thing as my used version, but makes use of identity e^(a logb) = b^a
Here, b = 10000, and a = -i/d_model, where i terms are from the arange

Both versions do the exact same thing. I used the pow version because it just felt simpler and has no drawbacks compared to the other version
'''


In [5]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout_p):
        super().__init__()
        
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionalFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, x, mask):
        attention_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attention_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

In [6]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, d_ff, num_heads, dropout_p):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionalFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_outputs = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_outputs))
        attn_outputs = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_outputs))
        ff_outputs = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_outputs))

        return x

In [7]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, num_layers, d_model, num_heads, d_ff, max_seq_length, dropout_p):
        super().__init__()
        
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length) 

        self.encoder_stack = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff, dropout_p) for _ in range(num_layers)])
        self.decoder_stack = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff, dropout_p) for _ in range(num_layers)])

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout_p)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        causal_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1)).bool()
        tgt_mask = tgt_mask & causal_mask

        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embeddings = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embeddings = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embeddings
        for enc_block in self.encoder_stack:
            enc_output = enc_block(enc_output, src_mask) # This encoder's output becomes next encoder block's output

        dec_output = tgt_embeddings
        for dec_block in self.decoder_stack:
            dec_output = dec_block(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output